# 文埋め込みを特徴量にして少ない学習データで分類する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/embedding_classifier.ipynb)

NLP クックブック 12-13 ／ テキスト分類

ラベル付きデータが各クラス数十件しかない。事前学習済みの文埋め込みを特徴量にして分類器を学習したい。

In [1]:
import io, tarfile, urllib.request
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor().sample(1800, random_state=0).reset_index(drop=True)
model = SentenceTransformer("intfloat/multilingual-e5-small")
emb = model.encode(["query: " + t for t in df["title"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
print(emb.shape)

(1800, 384)


In [2]:
pool_idx, test_idx = train_test_split(df.index, test_size=600, stratify=df["label"], random_state=0)
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True).fit(df["title"])
X_tfidf = tfidf.transform(df["title"])

for n_per_class in [4, 16, 64]:
    train_idx = df.loc[pool_idx].groupby("label").sample(n_per_class, random_state=0).index
    acc_emb = LogisticRegression(C=10, max_iter=1000).fit(emb[train_idx], df["label"][train_idx]).score(emb[test_idx], df["label"][test_idx])
    acc_tfidf = LogisticRegression(C=10, max_iter=1000).fit(X_tfidf[train_idx], df["label"][train_idx]).score(X_tfidf[test_idx], df["label"][test_idx])
    print(f"各クラス {n_per_class:3} 件: 埋め込み + LR = {acc_emb:.3f}   TF-IDF + LR = {acc_tfidf:.3f}")

各クラス   4 件: 埋め込み + LR = 0.558   TF-IDF + LR = 0.502
各クラス  16 件: 埋め込み + LR = 0.655   TF-IDF + LR = 0.650
各クラス  64 件: 埋め込み + LR = 0.757   TF-IDF + LR = 0.755


- 学習データが少ないほど、事前学習済みの埋め込みの効果が大きい。データが増えると TF-IDF との差は縮まる。
- 埋め込みは 1 回計算して保存しておけば、分類器の学習は一瞬で終わる。モデル本体を更新しないので GPU も不要。
- さらに精度を上げたいなら、対照学習で埋め込みモデル自体を少数例で微調整する SetFit がある。